In [158]:
from datetime import date, timedelta

from h2.utilities import guard_increment_window

today = date.today()
window1_start = today - timedelta(days=7)
window1_end = today
window2_start = today - timedelta(days=14)
window2_end = today - timedelta(days=8)

date_windows = [
    (window1_start.isoformat(), window1_end.isoformat()),
    (window2_start.isoformat(), window2_end.isoformat()),
]

In [159]:
import requests

API_KEY = "gGscoHYJYRJ0dJdEebDQp6uHAqP3SQaaVixYeZhw"
BASE_URL = "https://api.nasa.gov/neo/rest/v1/feed"

all_records = []

for start_date, end_date in date_windows:
    params = {
        "start_date": start_date,
        "end_date": end_date,
        "api_key": API_KEY,
    }
    try:
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()
        payload = response.json()
    except requests.exceptions.RequestException as e:
        print(f"API call failed for {start_date} to {end_date}: {e}")
        continue

    for date_str, objects_on_date in payload["near_earth_objects"].items():
        all_records.extend(objects_on_date)

print(f"Pulled {len(all_records)} total records across {len(date_windows)} windows.")

Pulled 61 total records across 2 windows.


In [160]:
from pathlib import Path

neo_ids = [str(obj["neo_reference_id"]) for obj in all_records]
neo_ids = list(dict.fromkeys(neo_ids))

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/raw/extracted_ids.txt").write_text("\n".join(neo_ids), encoding="utf8")

from generate_sentinel_log import generate_sentinel_log

generate_sentinel_log(neo_ids)

[generate_sentinel_log] 61 input ids -> 61 log rows (6 dropped, 6 ghost ids injected) -> data\raw\ground_station_log.csv


WindowsPath('data/raw/ground_station_log.csv')

In [161]:
cleaned = []
for record in all_records:
    if record["close_approach_data"]:
        cleaned.append(record)

In [162]:
def safe_float(value, default=None):
    try:
        return float(value)
    except (ValueError, TypeError):
        return default

In [163]:
def walk(value, path=""):
    if isinstance(value, dict):
        for key, v in value.items():
            walk(v, path + "/" + key)
    elif isinstance(value, list):
        if value:
            walk(value[0], path + "[0]")
    else:
        print(f"{path}: {type(value)}")


for record in all_records[:5]:
    print("--- record ---")
    walk(record)

--- record ---
/links/self: <class 'str'>
/id: <class 'str'>
/neo_reference_id: <class 'str'>
/name: <class 'str'>
/nasa_jpl_url: <class 'str'>
/absolute_magnitude_h: <class 'float'>
/estimated_diameter/kilometers/estimated_diameter_min: <class 'float'>
/estimated_diameter/kilometers/estimated_diameter_max: <class 'float'>
/estimated_diameter/meters/estimated_diameter_min: <class 'float'>
/estimated_diameter/meters/estimated_diameter_max: <class 'float'>
/estimated_diameter/miles/estimated_diameter_min: <class 'float'>
/estimated_diameter/miles/estimated_diameter_max: <class 'float'>
/estimated_diameter/feet/estimated_diameter_min: <class 'float'>
/estimated_diameter/feet/estimated_diameter_max: <class 'float'>
/is_potentially_hazardous_asteroid: <class 'bool'>
/close_approach_data[0]/close_approach_date: <class 'str'>
/close_approach_data[0]/close_approach_date_full: <class 'str'>
/close_approach_data[0]/epoch_date_close_approach: <class 'int'>
/close_approach_data[0]/relative_velocit

In [164]:
diameters = [r["estimated_diameter"]["kilometers"]["estimated_diameter_max"] for r in cleaned]
min_d = diameters[0]
max_d = diameters[0]
total_d = 0

for d in diameters:
    if d < min_d: min_d = d
    if d > max_d: max_d = d
    total_d += d
mean_d = total_d / len(diameters)
print(f"diameter: min={min_d} max={max_d} mean={mean_d}")

diameter: min=0.0059434687 max=7.9806814968 mean=0.36725959060819674


In [165]:
distances = [safe_float(r["close_approach_data"][0]["miss_distance"]["kilometers"]) for r in cleaned]
min_dist, max_dist, total_dist = distances[0], distances[0], 0
for d in distances:
    if d < min_dist:
        min_dist = d
    if d > max_dist:
        max_dist = d
    total_dist += d
mean_dist = total_dist / len(distances)
print(f"miss_distance_km: min={min_dist} max={max_dist} mean={mean_dist}")

miss_distance_km: min=3584832.90733736 max=74726688.69833517 mean=43443071.45283729


In [166]:
velocities = [safe_float(r["close_approach_data"][0]["relative_velocity"]["kilometers_per_hour"]) for r in cleaned]

min_v = velocities[0]
max_v = velocities[0]
total_v = 0

for v in velocities:
    if v < min_v:
        min_v = v
    if v > max_v:
        max_v = v
    total_v += v

mean_v = total_v / len(velocities)
print(f"min={min_v}, max={max_v}, mean={mean_v}")

min=7966.53716768, max=103523.4043964903, mean=48542.33392101342


In [167]:
mags = [record["absolute_magnitude_h"] for record in cleaned if record.get("absolute_magnitude_h") is not None]
mags_sorted = sorted(mags)
median_mag = mags_sorted[len(mags_sorted) // 2]

for record in cleaned:
    if record.get("absolute_magnitude_h") is None:
        record["absolute_magnitude_h"] = median_mag

In [168]:
total = len(all_records)
missing_close_approach = 0
missing_magnitude = 0
non_bool_hazard_flag = 0

for r in all_records:
    if not r["close_approach_data"]:
        missing_close_approach += 1
    if r.get("absolute_magnitude_h") is None:
        missing_magnitude += 1
    if not isinstance(r["is_potentially_hazardous_asteroid"], bool):
        non_bool_hazard_flag += 1

print(f"missing close_approach_data: {missing_close_approach}/{total} ({missing_close_approach / total * 100:.1f}%)")
print(f"missing absolute_magnitude_h: {missing_magnitude}/{total} ({missing_magnitude / total * 100:.1f}%)")
print(f"non-bool hazard flags: {non_bool_hazard_flag}")

missing close_approach_data: 0/61 (0.0%)
missing absolute_magnitude_h: 0/61 (0.0%)
non-bool hazard flags: 0


In [169]:
for r in cleaned:
    max_d = r["estimated_diameter"]["kilometers"]["estimated_diameter_max"]
    miss_lunar = safe_float(r["close_approach_data"][0]["miss_distance"]["lunar"])
    miss_km = safe_float(r["close_approach_data"][0]["miss_distance"]["kilometers"])
    velocity = safe_float(r["close_approach_data"][0]["relative_velocity"]["kilometers_per_hour"])
    distance = max_d / miss_lunar
    r["max_diameter_km"] = max_d
    r["miss_distance_km"] = miss_km
    r["miss_distance_lunar"] = miss_lunar
    r["relative_velocity_kph"] = velocity
    r["size_to_distance_ratio"] = distance

    if miss_lunar <= 5:
        r["approach_category"] = "very_close"
    elif miss_lunar <= 20:
        r["approach_category"] = "close"
    elif miss_lunar <= 60:
        r["approach_category"] = "moderate"
    else:
        r["approach_category"] = "distant"

    r["priority_watch"] = 1 if (max_d >= 0.14 and miss_lunar <= 10) else 0

In [170]:
import csv

with open("data/raw/ground_station_log.csv") as f:
    log_lookup = {row["neo_id"]: row for row in csv.DictReader(f)}

for r in cleaned:
    log_row = log_lookup.get(r["neo_reference_id"])
    r["observatory_code"] = log_row["observatory_code"] if log_row else None
    r["confidence_score"] = log_row["confidence_score"] if log_row else None

In [171]:
ratios = [record["size_to_distance_ratio"] for record in cleaned]
min_x, max_x = ratios[0], ratios[0]
for x in ratios:
    if x < min_x:
        min_x = x
    if x > max_x:
        max_x = x

for record in cleaned:
    record["scaled_size_to_distance_ratio"] = (record["size_to_distance_ratio"] - min_x) / (max_x - min_x)

In [172]:
from pathlib import Path

export_dir = Path("data/processed")
export_dir.mkdir(parents=True, exist_ok=True)
csv_path = Path("data/processed/clean_data.csv")
crosstab = {(True, True): 0, (True, False): 0, (False, True): 0, (False, False): 0}
for r in cleaned:
    key = (bool(r["priority_watch"]), r["is_potentially_hazardous_asteroid"])
    crosstab[key] += 1

print(crosstab)

import csv
Fieldnames = set()
for r in cleaned:
    Fieldnames.update(r.keys())
Fieldnames = list(Fieldnames)
Fieldnames = [
    "neo_reference_id", "name",
    "max_diameter_km", "miss_distance_km", "miss_distance_lunar",
    "relative_velocity_kph", "absolute_magnitude_h",
    "size_to_distance_ratio", "scaled_size_to_distance_ratio",
    "approach_category", "priority_watch",
    "is_potentially_hazardous_asteroid",
    "observatory_code", "confidence_score",
]
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=Fieldnames, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(cleaned)

{(True, True): 1, (True, False): 0, (False, True): 3, (False, False): 57}


In [173]:
n_total = len(cleaned)
n_flagged = sum(1 for r in cleaned if r["priority_watch"] == 1)
pct_workload_reduction = (1 - (n_flagged / n_total)) * 100
print(f"{pct_workload_reduction:.1f}% workload reduction")

98.4% workload reduction


In [174]:
import requests

url = "https://science.nasa.gov/science-research/planetary-science/planetary-defense/near-earth-asteroids/"
response = requests.get(url)
html = response.text

anchor = "Total number of discovered near-Earth asteroids"
idx = html.find(anchor)
window = html[idx - 40: idx]
print(repr(window).replace("'", ""))

2025.   Key statistics include: 39,123: 
